In [ ]:
import sys
print("Python:", sys.version)


In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import rcParams

seed = 9001
np.random.seed(seed)

### Create function to load individual patient data into one big dataframe

In [ ]:
regex = re.compile(r'\d+')

DATA_DIR = "/kaggle/input/training-sepsis-dataset"  # Path to the data

def load_single_file(file_path):
    df = pd.read_csv(file_path, sep='|')
    df['hour'] = np.arange(len(df))  # hour within each patient stay

    # Extract the patient number from the file name, e.g. p00001.psv -> 1.
    patient_nums = regex.findall(os.path.basename(file_path))
    df['patient'] = int(patient_nums[0]) if patient_nums else np.nan
    return df

def get_data_files():
    return sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.psv'), recursive=True))

def clean_data(data):
    data = data.reset_index(drop=True)
    return data

def load_data():
    df_list = []
    files = get_data_files()

    for i, file in enumerate(files):
        patient_df = load_single_file(file)
        patient_df['id'] = i   # Internal sequential patient ID starting from 0.
        df_list.append(patient_df)

    df = pd.concat(df_list, ignore_index=True)
    df = clean_data(df)

    # Sort each patient's records in temporal order.
    df = df.sort_values(['id', 'hour']).reset_index(drop=True)

    df = df.drop(columns=['patient'], errors='ignore')

    return df

In [ ]:
# aggregate the individual patient data into one big data frame
df = load_data()
print(df.shape)
print(df[['id', 'hour']].head(10))
print(df[['id', 'hour']].tail(10))
print(df.sort_values(['id', 'hour']).equals(df))

In [ ]:
sepsis_start = (
    df[df["SepsisLabel"] == 1]
    .groupby("id")["hour"]
    .min()
    .reset_index(name="sepsis_hour")
)

early_ids = sepsis_start[sepsis_start["sepsis_hour"] < 10]["id"]

pid = early_ids.iloc[0]
patient_df = df[df["id"] == pid].copy()
patient_df = patient_df[patient_df["hour"] <= 24]


In [ ]:
hours = patient_df["hour"]
hr = patient_df["HR"]
sepsis = patient_df["SepsisLabel"]

plt.figure(figsize=(8,4))

plt.scatter(hours, hr, color="blue", label="Heart Rate")
plt.scatter(hours[sepsis==1], hr[sepsis==1], color="red", label="septic hours")

plt.axvspan(0, 5, color="steelblue", alpha=0.4, label="6-hour sliding window")

plt.xlabel("Hour")
plt.ylabel("Heart Rate")
plt.title("6-hour sliding window")
plt.legend()
plt.show()


In [ ]:
for start in [0, 5, 10]:
    plt.figure(figsize=(8,4))

    plt.scatter(hours, hr, color="blue", label="Heart Rate")
    plt.scatter(hours[sepsis==1], hr[sepsis==1], color="red", label="septic hours")

    plt.axvspan(start, start+5, color="steelblue", alpha=0.4)

    plt.title(f"6-hour sliding window (hour {start} → {start+5})")
    plt.xlabel("Hour")
    plt.ylabel("Heart Rate")
    plt.legend()
    plt.show()


# Data Exploration

In [ ]:
df.columns # name of columns

In [ ]:
df.shape # dataset size: 1,552,210 rows x 43 columns

In [ ]:
df['SepsisLabel'].value_counts() # this is per event notes/records. NOT per patients

### Distribution of sepsis labels vs total number of events

In [ ]:
distribution = (df['SepsisLabel'].value_counts()[1] / sum(df['SepsisLabel'].value_counts()))
print('Distribution of sepsis events: ', distribution*100, '%')

### Plot the event-wise imbalance

In [ ]:
plt.bar(df['SepsisLabel'].value_counts().index, # x-values
        df['SepsisLabel'].value_counts()) # y-values
plt.xticks([0,1])
plt.xlabel('Class')
plt.ylabel('Frequency')
plt.title('Sepsis Label Distribution Event-wise')

### Distribution of sepsis patient-wise

In [ ]:
grp_sepsis = df.groupby(['id'])['SepsisLabel'].sum() # sum sepsis labels grouped by 'id'. patients with sepsis have sum > 0

len(grp_sepsis[grp_sepsis > 0]) # total number of patients with sepsis

In [ ]:
sepsis_distr = len(grp_sepsis[grp_sepsis > 0]) / len(df['id'].unique())
print('Distribution of sepsis patients: ', sepsis_distr*100, '%')

### Percentage of missing data per each variable

In [ ]:
missing = (df.isnull().sum() / df.shape[0]) * 100
missing_sorted = missing.sort_values() # sorted percentage of missing data variables
missing_sorted

In [ ]:
# Keep a broad set of raw columns following the challenge feature set.
RAW_KEEP_COLS = [
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2',
    'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2',
    'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
    'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate',
    'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT',
    'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender',
    'HospAdmTime', 'ICULOS'
]

BASE_COLS = ['id', 'hour', 'SepsisLabel']

selected_cols = [col for col in BASE_COLS + RAW_KEEP_COLS if col in df.columns]
selected_cols

### Exclude Unit1, Unit2, variables because these are just identifiers

In [ ]:
selected_cols = [col for col in selected_cols if col not in ['Unit1', 'Unit2']]

In [ ]:
selected_cols

In [ ]:
filter_df = df[selected_cols].copy()
filter_df.shape

In [ ]:
filter_df.shape # 1,552,210 rows x 15 variables

In [ ]:
filter_df.head()

In [ ]:
categorical = filter_df.dtypes == object
categorical['Gender'] = True  # treat Gender as categorical

In [ ]:
cat_vars = list(categorical[categorical].index)
cont_vars = list(categorical[~categorical].index)

# Exclude ID, time, and label columns from the continuous signal group.
cont_vars = [var for var in cont_vars if var not in ['id', 'hour', 'SepsisLabel']]

cat_vars, cont_vars[:10], len(cont_vars)

In [ ]:
cont_vars

### Boxplot of different variables between sepsis and non-sepsis patients

In [ ]:
for i, var in enumerate(cont_vars):
  plt.figure(i)
  ax = sns.boxplot(x="SepsisLabel", y=var, data=filter_df)

### Base-stage missing handling

At this stage, only a small set of stable or background variables is lightly imputed, including Age, Gender, HospAdmTime, and ICULOS. Most dynamic clinical measurements are intentionally left missing at this stage so that missingness patterns can be preserved for downstream feature engineering and sequence preparation.

In [ ]:
# Base-stage imputation for a small set of stable/background variables
X = filter_df.copy()
X = X.sort_values(['id', 'hour']).reset_index(drop=True)

if 'Age' in X.columns:
    X['Age'] = X['Age'].fillna(X['Age'].median())

if 'Gender' in X.columns:
    mode_gender = X['Gender'].mode(dropna=True)
    if len(mode_gender) > 0:
        X['Gender'] = X['Gender'].fillna(mode_gender.iloc[0])

if 'HospAdmTime' in X.columns:
    X['HospAdmTime'] = X.groupby('id')['HospAdmTime'].ffill().bfill()
    X['HospAdmTime'] = X['HospAdmTime'].fillna(X['HospAdmTime'].median())

if 'ICULOS' in X.columns:
    X['ICULOS'] = X.groupby('id')['ICULOS'].ffill().bfill()

print(X.shape)
print((X.isnull().sum() / len(X) * 100).sort_values(ascending=False).head(20))

### Remaining missing values are intentionally preserved

Many dynamic clinical variables still remain missing after base-stage cleaning. This is intentional, as the missingness pattern will be handled later during feature engineering and sequence preparation.

In [ ]:
missing = (X.isnull().sum() / X.shape[0]) * 100
missing

### Save this processed data for later access

In [ ]:
X.to_csv('stage1_base_clean.csv', index=False)

In [ ]:
signals = ['HR', 'MAP', 'O2Sat', 'SBP', 'Resp', 'DBP', 'Temp', 'id', 'hour', 'SepsisLabel']

In [ ]:
other_cols = [var for var in X.columns if var not in signals]

In [ ]:
other_cols

### Return some ID's of the patients that get sepsis

In [ ]:
sepsis_id = X.loc[X['SepsisLabel'] == 1]['id'].unique()

In [ ]:
# returns the first 10 patients with sepsis
sepsis_id[:20]

### Plot a sample Heart Rate between a sepsis and non-sepsis patient to see the difference in pattern over time

In [ ]:
rcParams['figure.figsize'] = 10, 10

In [ ]:
# # Function to map the colors as a list from the input list of x variables
colors = {1: 'red', 0: '#1f77b4'}

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

ax1.scatter(X[X['id']==52]['hour'].values, X[X['id']==52]['HR'], c = X[X['id']==52]['SepsisLabel'].apply(lambda x: colors[x]))
ax1.set_xlabel('Hour')
ax1.set_ylabel('Heart Rate')
ax1.set_title('Heart Rate of A Sepsis Patient Over Time')

ax2.scatter(X[X['id']==95]['hour'].values, X[X['id']==95]['HR'])
ax2.set_xlabel('Hour')
ax2.set_ylabel('Heart Rate')
ax2.set_ylim([44, 58])
ax2.set_title('Heart Rate of A Non-Sepsis Patient Over Time')

### Plot Mean Arterial Pressure (MAP) of the same sepsis and non-sepsis patient

In [ ]:
rcParams['figure.figsize'] = 10, 10

In [ ]:
# Map sepsis labels to plot colors.
colors = {1: 'red', 0: '#1f77b4'}

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

ax1.scatter(X[X['id']==52]['hour'].values, X[X['id']==52]['MAP'], c = X[X['id']==52]['SepsisLabel'].apply(lambda x: colors[x]))
ax1.set_xlabel('Hour')
ax1.set_ylabel('Mean Arterial Pressure')
ax1.set_title('Mean Arterial Pressure of A Sepsis Patient Over Time')


ax2.scatter(X[X['id']==95]['hour'].values, X[X['id']==95]['MAP'])
ax2.set_xlabel('Hour')
ax2.set_ylabel('Mean Arterial Pressure')
ax2.set_ylim([86, 113])
ax2.set_title('Mean Arterial Pressure of A Non-Sepsis Patient Over Time')